In [1]:
%pip install geopandas networkx xarray rioxarray exactextract netcdf4 pyogrio

# %% [markdown]
# # Series de precipitación por pozo: cuenca + cuencas aguas arriba (HydroBASINS lev05 a lev12)
# Reemplaza el notebook anterior (CAMELS/BNA). Sin evapotranspiración.

# %%
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import box
import rasterio
from exactextract import exact_extract

# ====== PARÁMETROS ======
root_data = '../../data/'
root_gis = r"C:/DOCUMENTS/EARTH_SCIENCES/CLIMA/2027_Alvarez-Garreton/R_nue1/GIS"
# GeoTIFF generado en R (carpeta ...\\R_nue1\\data). Copialo a GIS o cambiá esta ruta.
tif_path = os.path.join(root_gis, "CR2MET_pr_v2.5_pr_mon_1960_2024_005deg.tif")
n_meses = 780              # 1960-01 a 2024-12
levels = range(7, 12)      # hybas_sa_lev05_v1c ... hybas_sa_lev12_v1c
min_obs = 100
lon_min, lon_max, lat_min, lat_max = -74, -66, -43, -17
tol_area = 0.01            # tolerancia relativa área del grafo vs UP_AREA

out_dir = os.path.join(root_data, "ms_data")
os.makedirs(out_dir, exist_ok=True)
mask_path = os.path.join(out_dir, "_pr_valid_mask.tif")

# %% [markdown]
# ## 1. Pozos (mismos filtros que antes)

# %%
gw_all = pd.read_csv(root_data + 'cr2sub/cr2sub_v1.1_gwl_mon_clean.csv', parse_dates=['date'])
att = pd.read_csv(root_data + 'cr2sub/cr2sub_v1.1_attributes_utm_nue.csv')
att['cr2sub_id'] = att['cr2sub_id'].astype(str)
att = att.set_index('cr2sub_id')

clean_codes = [c for c in gw_all.columns if c != 'date']
keep = []
for cod in clean_codes:
    if cod not in att.index:
        print(f"⚠️ Attributes not available for {cod}. Skipping.")
        continue
    if gw_all[cod].notna().sum() < min_obs:
        continue
    if att.loc[cod, ['cr2sub_lat', 'cr2sub_lon']].isna().any():
        print(f"⚠️ Sin coordenadas para {cod}. Skipping.")
        continue
    keep.append(cod)

wells = gpd.GeoDataFrame(
    {'cr2sub_id': keep},
    geometry=gpd.points_from_xy(att.loc[keep, 'cr2sub_lon'], att.loc[keep, 'cr2sub_lat']),
    crs="EPSG:4326",
)
print(f"{len(keep)} pozos de {len(clean_codes)}")

# GW: igual que antes (la tabla ya trae todos los meses 1960-2024, con NaN)
wells_gw = gw_all.set_index('date')[keep].sort_index()
wells_gw_an = wells_gw - wells_gw.groupby(wells_gw.index.month).transform('mean')

# %% [markdown]
# ## 2. Grilla de precipitación: GeoTIFF multibanda generado en R (780 meses, 1960-01 a 2024-12)

# %%
with rasterio.open(tif_path) as src:
    assert src.count == n_meses, f"El tif tiene {src.count} bandas, se esperaban {n_meses}"
    a1 = src.read(1, masked=True)
    print(f"tif: bandas={src.count}, shape={a1.shape}, crs={src.crs}, nodata={src.nodata}")
    print(f"banda 1: celdas con dato={a1.count()}, min={a1.min():.2f}, max={a1.max():.2f}")
    print(f"bounds={src.bounds}")
    assert a1.count() > 0, "La banda 1 no tiene datos"
    # Máscara de cobertura (misma en todo el período): 1 = hay dato, 0 = no
    prof = src.profile.copy()
    prof.update(count=1, dtype='float32', nodata=None, compress='deflate')
    with rasterio.open(mask_path, 'w', **prof) as dst:
        dst.write((~np.ma.getmaskarray(a1)).astype('float32'), 1)

# Fechas de las bandas: enero 1960 en adelante, un mes por banda
tiempos = pd.date_range("1960-01-01", periods=n_meses, freq="MS")


# %% [markdown]
# ## 3. Series por nivel

# %%
bbox = (lon_min, lat_min, lon_max, lat_max)
gw_dates = wells_gw.index
gw_periods = gw_dates.to_period('M')


def serie_nivel(lvl):
    shp = os.path.join(root_gis, f"hybas_sa_lev{lvl:02d}_v1c.shp")
    try:
        b = gpd.read_file(shp, bbox=bbox)
    except Exception as e:   # índice .sbn corrupto: leer todo y filtrar después
        print(f"   bbox falló ({str(e)[:50]}...), leyendo completo y filtrando")
        b = gpd.read_file(shp)
        b = b[b.intersects(box(*bbox))]
    if b.crs is None or b.crs.to_epsg() != 4326:
        b = b.to_crs(4326)
    b['HYBAS_ID'] = b['HYBAS_ID'].astype('int64')
    b = b.drop_duplicates('HYBAS_ID')

    # Grafo: h -> NEXT_DOWN (ancestros de h = lo que drena hacia h)
    G = nx.DiGraph()
    G.add_nodes_from(b.HYBAS_ID)
    G.add_edges_from((h, n) for h, n in zip(b.HYBAS_ID, b.NEXT_DOWN) if n != 0 and n in G)

    # Pozo -> cuenca
    pj = gpd.sjoin(wells, b[['HYBAS_ID', 'geometry']], how='left', predicate='within')
    pj = pj.drop_duplicates('cr2sub_id')
    sin = pj.loc[pj.HYBAS_ID.isna(), 'cr2sub_id'].tolist()
    print(f"Nivel {lvl:02d}: {len(b)} cuencas en bbox, {len(sin)} pozos sin cuenca {sin[:10]}")
    pj = pj.dropna(subset=['HYBAS_ID']).copy()
    pj['HYBAS_ID'] = pj.HYBAS_ID.astype('int64')

    # Cuenca + aguas arriba
    ups = {h: nx.ancestors(G, h) | {h} for h in pj.HYBAS_ID.unique()}
    necesarias = set().union(*ups.values())
    sub = b[b.HYBAS_ID.isin(necesarias)]

    # P por cuenca individual (media ponderada por cobertura de celda; ignora NaN)
    # (sin include_cols: los HYBAS_ID no caben en int32; las filas salen en el mismo orden que sub)
    sub = sub[['HYBAS_ID', 'geometry']].reset_index(drop=True)
    r = exact_extract(tif_path, sub[['geometry']], ['mean'], output='pandas')
    assert len(r) == len(sub) and r.shape[1] == len(tiempos), "filas/bandas no coinciden"
    Ploc = pd.DataFrame(r.values, index=sub.HYBAS_ID.values, columns=tiempos)
    rc = exact_extract(mask_path, sub[['geometry']], ['mean'], output='pandas')
    cov = pd.Series(rc.iloc[:, 0].values, index=sub.HYBAS_ID.values)
    print(f"   [diag] Ploc: % no-NaN={Ploc.notna().values.mean():.1%}, "
          f"media={np.nanmean(Ploc.values):.2f}")
    print(f"   cuencas con cobertura parcial de P: {(cov < 0.999).sum()} "
          f"(sin dato: {(cov < 0.001).sum()})")

    area = b.set_index('HYBAS_ID')['SUB_AREA']
    up_area = b.set_index('HYBAS_ID')['UP_AREA']

    # Promedio ponderado por SUB_AREA; si falta dato se renormaliza con las cuencas que tienen
    rows, chk = {}, []
    for h, ids in ups.items():
        ids = list(ids)
        V = Ploc.loc[ids].values
        a = area.loc[ids].values[:, None] * np.ones_like(V)
        a = np.where(np.isnan(V), 0.0, a)
        with np.errstate(invalid='ignore', divide='ignore'):
            rows[h] = np.nansum(V * a, axis=0) / a.sum(axis=0)
        chk.append((h, area.loc[ids].sum(), up_area.loc[h]))
    Pup = pd.DataFrame(rows, index=tiempos).T
    print(f"   [diag] Pup: % no-NaN={Pup.notna().values.mean():.1%}")

    chk = pd.DataFrame(chk, columns=['HYBAS_ID', 'area_grafo', 'UP_AREA'])
    chk['dif_rel'] = (chk.area_grafo - chk.UP_AREA).abs() / chk.UP_AREA
    malas = chk[chk.dif_rel > tol_area]
    print(f"   check área grafo vs UP_AREA: max dif = {chk.dif_rel.max():.2%}, "
          f"{len(malas)} cuencas fuera de tolerancia")
    if len(malas):
        pozos_mal = pj[pj.HYBAS_ID.isin(malas.HYBAS_ID)].cr2sub_id.tolist()
        print(f"   pozos afectados (aguas arriba incompleto): {pozos_mal[:10]}")

    # Serie por pozo, alineada al índice mensual de GW
    pr_w = Pup.loc[pj.HYBAS_ID.values].T
    pr_w.columns = pj.cr2sub_id.values
    pr_w.index = pr_w.index.to_period('M')
    pr_w = pr_w.reindex(gw_periods)
    pr_w.index = gw_dates
    pr_w.index.name = 'date'
    print(f"   [diag] pr_w final: % no-NaN={pr_w.notna().values.mean():.1%}, "
          f"fechas {pr_w.index.min().date()} -> {pr_w.index.max().date()}")

    # Metadatos
    meta = pd.DataFrame({
        'cr2sub_id': pj.cr2sub_id.values,
        'HYBAS_ID': pj.HYBAS_ID.values,
        'HYBAS_ID_upstream': [';'.join(str(i) for i in
                                       [h] + sorted(ups[h] - {h})) for h in pj.HYBAS_ID],
        'UP_AREA': up_area.loc[pj.HYBAS_ID].values,
        'cobertura_P': cov.loc[pj.HYBAS_ID].values,
    })
    return pr_w, meta


for lvl in levels:
    pr_w, meta = serie_nivel(lvl)
    pr_an = pr_w - pr_w.groupby(pr_w.index.month).transform('mean')   # 1960-2024 completo
    pr_w.to_csv(os.path.join(out_dir, f"wells_pr_lev{lvl:02d}.csv"))
    pr_an.to_csv(os.path.join(out_dir, f"wells_pr_an_lev{lvl:02d}.csv"))
    meta.to_csv(os.path.join(out_dir, f"wells_meta_lev{lvl:02d}.csv"), index=False)

# GW (no depende de la cuenca)
wells_gw.to_csv(os.path.join(out_dir, "wells_gw.csv"))
wells_gw_an.to_csv(os.path.join(out_dir, "wells_gw_an.csv"))
print("Finished!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


506 pozos de 1137
tif: bandas=780, shape=(800, 220), crs=EPSG:4326, nodata=nan
banda 1: celdas con dato=39702, min=0.00, max=432.21
bounds=BoundingBox(left=-77.0, bottom=-57.0, right=-66.00000000000001, top=-17.0)
Nivel 05: 66 cuencas en bbox, 0 pozos sin cuenca []
   [diag] Ploc: % no-NaN=100.0%, media=25.50
   cuencas con cobertura parcial de P: 2 (sin dato: 0)
   [diag] Pup: % no-NaN=100.0%
   check área grafo vs UP_AREA: max dif = 0.01%, 0 cuencas fuera de tolerancia
   [diag] pr_w final: % no-NaN=95.9%, fechas 1957-04-01 -> 2024-12-01
Nivel 06: 224 cuencas en bbox, 0 pozos sin cuenca []
   [diag] Ploc: % no-NaN=100.0%, media=18.76
   cuencas con cobertura parcial de P: 2 (sin dato: 0)
   [diag] Pup: % no-NaN=100.0%
   check área grafo vs UP_AREA: max dif = 0.11%, 0 cuencas fuera de tolerancia
   [diag] pr_w final: % no-NaN=95.9%, fechas 1957-04-01 -> 2024-12-01
Nivel 07: 746 cuencas en bbox, 0 pozos sin cuenca []
   [diag] Ploc: % no-NaN=100.0%, media=22.02
   cuencas con cobertur